In [2]:
import json
import os

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv

from utils.data import edinet_to_industry_map, all_securities, jp_500
from utils.datetime import date_string_to_quarter
from utils.edinet_api import get_doc_name
from utils.ELO import EloRatingSystem, get_num_games, generate_random_pdf_pair
from utils.signal import get_winner, get_winner_grok_mini, get_stock_code_from_path

from utils.datapath import (
    documents_path,
    edinet_codes_path,
    docs_metadata_path,
    industry_elo_signals_path
)

In [3]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

grok_client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

In [4]:
df = pd.read_excel(edinet_codes_path)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [5]:
top_security_codes = list(set([code for quarter, codes in jp_500.items() for code in codes]))
top_securities = [security for security in all_securities if security["code"] in top_security_codes]
top_securities_edinet = [security["edinet_code"] for security in top_securities]
len(top_securities)

740

In [6]:
filtered_doc_metadata = [doc for doc in docs_metadata if doc["edinetCode"] in top_securities_edinet]
len(filtered_doc_metadata)

20175

In [7]:
industry_quarterly_docs = defaultdict(lambda: defaultdict(list))

for doc in filtered_doc_metadata:
    industry = edinet_to_industry_map.get(doc['edinetCode'], "na")
    period_end_quater = date_string_to_quarter(doc["periodEnd"])

    industry_path = os.path.join(documents_path, industry)
    quarter_path = os.path.join(industry_path, period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(quarter_path, save_name)

    industry_quarterly_docs[industry][period_end_quater].append(output_path)

In [8]:
selected_industries = [
    # "Real Estate",
    # "Precision Instruments",
    # "Electric Power & Gas",
    # "Other Financing Business",
    # "Glass & Ceramics Products",
    # "Nonferrous Metals",
    # "Securities & Commodity Futures",
    "Iron & Steel",
    "Metal Products",
]
# selected_industries = industry_quarterly_docs.keys()

In [9]:
industry_quarterly_signals = {}

In [10]:
industry = "Iron & Steel"
quarterly_docs = industry_quarterly_docs[industry]

client = grok_client
win_fn = get_winner_grok_mini

iron_n_steel_elo_system = EloRatingSystem(ratings={})
quarterly_signals = {}

# Calculate total iterations (quarters * games per quarter)
total_games = sum(get_num_games(len(docs)) for docs in quarterly_docs.values())
progress_bar = tqdm(total=total_games, desc="Playing games")

for quarter, docs in quarterly_docs.items():
    for pdf1_path, pdf2_path in generate_random_pdf_pair(
        docs, get_num_games(len(docs))
    ):
        try:
            winner = win_fn(client, pdf1_path, pdf2_path)
        except:
            print(f"Grok failed for {pdf1_path} vs {pdf2_path}")

        companyA_code, companyB_code = [
            get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
        ]
        iron_n_steel_elo_system.update_ratings(companyA_code, companyB_code, winner)
        progress_bar.update(1)

    quarterly_signals[quarter] = iron_n_steel_elo_system.get_all_ratings()

progress_bar.close()
industry_quarterly_signals[industry] = quarterly_signals

Playing games:  19%|█▉        | 146/772 [19:25<1:27:20,  8.37s/it]

Grok failed for ../../documents/Iron & Steel/2016-Q4/E01264_ＪＦＥホールディングス株式会社_140_S1009JU0.pdf vs ../../documents/Iron & Steel/2016-Q4/E01253_丸一鋼管株式会社_140_S1009MK1.pdf


Playing games:  24%|██▎       | 182/772 [24:07<1:14:28,  7.57s/it]

Grok failed for ../../documents/Iron & Steel/2017-Q2/E01231_株式会社　神戸製鋼所_140_S100B056.pdf vs ../../documents/Iron & Steel/2017-Q2/E01225_日本製鉄株式会社_140_S100B2H0.pdf


Playing games:  34%|███▎      | 259/772 [34:28<1:25:54, 10.05s/it]

...

Playing games:  60%|█████▉    | 463/772 [1:02:17<41:23,  8.04s/it]

Grok failed for ../../documents/Iron & Steel/2020-Q3/E01264_ジェイ　エフ　イー　ホールディングス株式会社_140_S100K23U.pdf vs ../../documents/Iron & Steel/2020-Q3/E01231_株式会社　神戸製鋼所_140_S100K0TN.pdf


Playing games:  93%|█████████▎| 719/772 [1:38:41<07:09,  8.11s/it]

Grok failed for ../../documents/Iron & Steel/2023-Q3/E01253_丸一鋼管株式会社_140_S100S901.pdf vs ../../documents/Iron & Steel/2023-Q3/E01259_大和工業株式会社_140_S100S9SC.pdf


Playing games: 100%|██████████| 772/772 [1:46:42<00:00,  8.29s/it]


In [11]:
industry = "Metal Products"
quarterly_docs = industry_quarterly_docs[industry]

client = grok_client
win_fn = get_winner_grok_mini

metal_products_elo_system = EloRatingSystem(ratings={})
quarterly_signals = {}

# Calculate total iterations (quarters * games per quarter)
total_games = sum(get_num_games(len(docs)) for docs in quarterly_docs.values())
progress_bar = tqdm(total=total_games, desc="Playing games")

for quarter, docs in quarterly_docs.items():
    for pdf1_path, pdf2_path in generate_random_pdf_pair(
        docs, get_num_games(len(docs))
    ):
        try:
            winner = win_fn(client, pdf1_path, pdf2_path)
        except:
            print(f"Grok failed for {pdf1_path} vs {pdf2_path}")

        companyA_code, companyB_code = [
            get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
        ]
        metal_products_elo_system.update_ratings(companyA_code, companyB_code, winner)
        progress_bar.update(1)

    quarterly_signals[quarter] = metal_products_elo_system.get_all_ratings()

progress_bar.close()
industry_quarterly_signals[industry] = quarterly_signals

Playing games:  55%|█████▍    | 290/528 [44:33<30:38,  7.73s/it]    

Grok failed for ../../documents/Metal Products/2020-Q2/E01353_東洋製罐グループホールディングス株式会社_140_S100JIK5.pdf vs ../../documents/Metal Products/2020-Q2/E01317_株式会社ＬＩＸＩＬグループ_140_S100JKOD.pdf


Playing games:  57%|█████▋    | 303/528 [46:10<27:46,  7.41s/it]

Grok failed for ../../documents/Metal Products/2020-Q2/E01353_東洋製罐グループホールディングス株式会社_140_S100JIK5.pdf vs ../../documents/Metal Products/2020-Q2/E01317_株式会社ＬＩＸＩＬグループ_140_S100JKOD.pdf


Playing games:  97%|█████████▋| 510/528 [1:12:51<02:16,  7.57s/it]

Grok failed for ../../documents/Metal Products/2023-Q4/E01353_東洋製罐グループホールディングス株式会社_140_S100SVWM.pdf vs ../../documents/Metal Products/2023-Q4/E01317_株式会社ＬＩＸＩＬ_140_S100SSJH.pdf


Playing games: 100%|██████████| 528/528 [1:15:19<00:00,  8.56s/it]


In [12]:
for industry, quarterly_docs in tqdm(industry_quarterly_docs.items()):
    # client = None if industry not in selected_industries else grok_client
    # win_fn = get_winner if industry not in selected_industries else get_winner_grok_mini

    if industry in selected_industries:
        continue

    client = None
    win_fn = get_winner
    
    elo_system = EloRatingSystem(ratings={})
    quarterly_signals = {}

    for quarter, docs in quarterly_docs.items():
        for pdf1_path, pdf2_path in generate_random_pdf_pair(
            docs, get_num_games(len(docs))
        ):
            winner = win_fn(client, pdf1_path, pdf2_path)
            companyA_code, companyB_code = [
                get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
            ]
            elo_system.update_ratings(companyA_code, companyB_code, winner)

        quarterly_signals[quarter] = elo_system.get_all_ratings()
    
    industry_quarterly_signals[industry] = quarterly_signals


100%|██████████| 33/33 [00:01<00:00, 27.22it/s]


In [13]:
# industry_quarterly_signals.get("Metal Products")
industry_quarterly_signals.get("Iron & Steel")
# industry_quarterly_signals.get("Marine Transportation")


{'2015-Q2': {'5463': 1516.3324935599205,
  '5444': 1651.2297000798205,
  '5471': 1534.6119166358326,
  '5423': 1455.033506888577,
  '5401': 1444.2630630505557,
  '5411': 1481.2891484684571,
  '5481': 1415.7285126878994,
  '5406': 1501.511658628937},
 '2015-Q3': {'5463': 1628.7084525043153,
  '5444': 1549.2949984188772,
  '5471': 1550.5227347438088,
  '5423': 1420.1705972406107,
  '5401': 1475.4533483140701,
  '5411': 1481.6329168992215,
  '5481': 1332.426989655615,
  '5406': 1561.7899622234813},
 '2015-Q4': {'5463': 1511.3523727435017,
  '5444': 1583.1016980877089,
  '5471': 1553.9644688613917,
  '5423': 1337.2193789239254,
  '5401': 1477.0810079441744,
  '5411': 1546.5449715696213,
  '5481': 1330.4604440945968,
  '5406': 1660.2756577750797},
 '2016-Q2': {'5463': 1490.5755742255858,
  '5444': 1516.8333542640821,
  '5471': 1433.5286380995233,
  '5423': 1338.3110342381651,
  '5401': 1566.9431331785927,
  '5411': 1573.1169357483645,
  '5481': 1396.3303690766952,
  '5406': 1684.36096116899

In [ ]:
# request_count

# Cost Analysis
model_input_prices = {
    "grok-3-mini": 0.3,
    "grok-3": 3,
    "deepseek-chat": 0.27,
    "deepseek-chat-discount": 0.135,
    "deepseek-reasoner": 0.55,
    "deepseek-reasoner-discount": 0.135,
}

num_requests = 0
model = "grok-3-mini"
num_input_token_per_request = 75000
price_per_million_input_token = model_input_prices[model] # US dollars 
price_per_request = (num_input_token_per_request / 1e6) * price_per_million_input_token
total_price = price_per_request * num_requests
print(f"Total cost per run: ${total_price:.2f}")

Total cost per run: $29.25


In [58]:
num_requests

1300

In [ ]:
# Industry wise elo - DONE
# Industry wise portfolio backtest - DONE

# But that's not the important point 
# Get to the api calls!!!!!!!!!

In [14]:
# Industry wise elo

with open(industry_elo_signals_path, 'w', encoding='utf-8') as f:
    json.dump(industry_quarterly_signals, f)

In [ ]:
# Deepseek >50% off from 12:30 to 8:30 am HKT